<a href="https://colab.research.google.com/github/thuynguyenhuit/hocsau/blob/main/Nhan_dien_benh_phoi_bang_mo_hinh_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive
import os

# 2. Định nghĩa đường dẫn (Hãy thay đổi tên folder cho đúng với Drive của bạn)
# Giả sử folder 'chest_xray' nằm ngay ngoài cùng của My Drive
base_dir = '/content/drive/MyDrive/CNN/chest_xray'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# 3. Cấu hình ImageDataGenerator với validation_split
# Chúng ta sẽ thực hiện Augmentation (tăng cường dữ liệu) cho tập Train
train_datagen = ImageDataGenerator(
    rescale=1./255,            # Chuẩn hóa pixel về [0, 1]
    validation_split=0.2,      # QUAN TRỌNG: Chia 20% dữ liệu để làm Validation
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Tập Test chỉ cần rescale, không cần split hay augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

# 4. Tạo các bộ nạp dữ liệu (Generators)

# Bộ nạp cho tập TRAIN (80% của thư mục train)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training',         # Xác định đây là phần training (80%)
    seed=123                   # Giữ seed cố định để việc chia file không bị xáo trộn mỗi lần chạy
)

# Bộ nạp cho tập VALIDATION (20% còn lại của thư mục train)
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation',       # Xác định đây là phần validation (20%)
    seed=123
)

# Bộ nạp cho tập TEST (Giữ nguyên từ thư mục test)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

# Kiểm tra nhãn lớp
print("Nhãn của các lớp:", train_generator.class_indices)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Nhãn của các lớp: {'NORMAL': 0, 'PNEUMONIA': 1}


In [6]:
from tensorflow.keras import layers, models

# Khởi tạo mô hình tuần tự
model = models.Sequential([
    # Khối 1: Trích xuất các đặc trưng cơ bản (đường nét)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),

    # Khối 2: Trích xuất đặc trưng phức tạp hơn
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    # Khối 3:
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    # Chuyển đổi dữ liệu từ dạng ảnh sang dạng phẳng (vector)
    layers.Flatten(),

    # Lớp Dense (Fully Connected) để phân loại
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5), # Chống học vẹt (overfitting)
    layers.Dense(1, activation='sigmoid') # 1 đầu ra: 0 (Normal) hoặc 1 (Pneumonia)
])


In [7]:
from tensorflow.keras import optimizers

# 1. Khai báo siêu tham số (Hyper-parameters)
LR = 0.001          # Tỷ lệ học (Learning Rate)
BATCH_SIZE = 32     # Số lượng mẫu dữ liệu trong một lần cập nhật trọng số
EPOCHS = 30         # Tổng số vòng lặp huấn luyện
OPTIMIZER = optimizers.Adam(learning_rate=LR)
LOSS_FUNCTION = 'binary_crossentropy'

# 2. Biên dịch mô hình (Compile)
model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS_FUNCTION,
    metrics=['accuracy']
)

print("Đã cấu hình xong siêu tham số. Sẵn sàng huấn luyện!")

Đã cấu hình xong siêu tham số. Sẵn sàng huấn luyện!


In [9]:
# Huấn luyện
history = model.fit(
    train_generator,
    epochs=EPOCHS, # Sử dụng biến EPOCHS đã khai báo ở trên
    validation_data=validation_generator
)

Epoch 1/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 776s 6s/step - accuracy: 0.8042 - loss: 0.4516 - val_accuracy: 0.8651 - val_loss: 0.3051
Epoch 2/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 359s 3s/step - accuracy: 0.8894 - loss: 0.2586 - val_accuracy: 0.9340 - val_loss: 0.1702
Epoch 3/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 353s 3s/step - accuracy: 0.8901 - loss: 0.2682 - val_accuracy: 0.9014 - val_loss: 0.2356
Epoch 4/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 352s 3s/step - accuracy: 0.9042 - loss: 0.2256 - val_accuracy: 0.9359 - val_loss: 0.1606
Epoch 5/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 347s 3s/step - accuracy: 0.9198 - loss: 0.2042 - val_accuracy: 0.9368 - val_loss: 0.1467
Epoch 6/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 358s 3s/step - accuracy: 0.9190 - loss: 0.2024 - val_accuracy: 0.9435 - val_loss: 0.1506
Epoch 7/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 357s 3s/step - accuracy: 0.9334 - loss: 0.1840 - val_accuracy: 0.9455 - val_loss: 0.1367
Epoch 8/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 364s 3s/step - accuracy: 0.9372 - loss: 0.1643 - val_accu